---
## Section 12 — Evaluation
**Work Package: Performance Evaluation**

**Method 1:** Holdout 80/20 — Precision@k, Recall@k, F1@k  
**Method 2:** Leave-one-out — Hit@10

Key fix: use `df_inter['recipe_id'].unique()` (original IDs) not `unique_recipes` (mapped indices).


In [ ]:
# ── Rebuild everything from scratch to guarantee consistency ─────────────────
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as svd_split

def prec_at_k(rec, rel, k): return len(set(rec[:k]) & rel) / k if k else 0
def rec_at_k(rec, rel, k):  return len(set(rec[:k]) & rel) / len(rel) if rel else 0
def f1(p, r):                return 2*p*r/(p+r) if p+r else 0

# Step 1 — Fresh mappings from df_inter
print('Step 1 — Building mappings...')
_all_users   = df_inter['user_id'].unique()
_all_recipes = df_inter['recipe_id'].unique()   # original recipe IDs
_u2i = {u: i for i, u in enumerate(_all_users)}
_r2i = {r: i for i, r in enumerate(_all_recipes)}
print(f'  Users:   {len(_u2i):,}')
print(f'  Recipes: {len(_r2i):,}')


In [ ]:
# Step 2 — Train SVD on these exact mappings
print('Step 2 — Training SVD...')
_df = df_inter[['user_id','recipe_id','rating']].copy()
_df['uid'] = _df['user_id'].map(_u2i)
_df['rid'] = _df['recipe_id'].map(_r2i)

_reader = Reader(rating_scale=(1, 5))
_data   = Dataset.load_from_df(_df[['uid','rid','rating']], _reader)
_train, _test = svd_split(_data, test_size=0.2, random_state=42)

_svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.1, random_state=42)
_svd.fit(_train)
_preds = _svd.test(_test)
print(f'  RMSE: {accuracy.rmse(_preds, verbose=False):.4f}')
print(f'  MAE:  {accuracy.mae(_preds,  verbose=False):.4f}')


In [ ]:
# Step 3 — Build relevance sets (ratings >= 4 = liked)
print('Step 3 — Building relevance sets...')
_counts     = df_inter.groupby('user_id').size()
_eval_uids  = _counts[_counts >= 5].index.tolist()

_relev = {}
for uid in _eval_uids:
    liked = set(df_inter[
        (df_inter['user_id'] == uid) &
        (df_inter['rating']  >= 4)
    ]['recipe_id'].values)
    if liked:
        _relev[uid] = liked

print(f'  Eval users with liked items: {len(_relev):,}')

# Sanity check
_s_uid  = list(_relev.keys())[0]
_s_rel  = _relev[_s_uid]
_s_seen = set(df_inter[df_inter['user_id']==_s_uid]['recipe_id'].values)
_s_unseen = [r for r in _all_recipes if r not in _s_seen and r in _r2i]
print(f'\n  Sanity check — user {_s_uid}:')
print(f'    In _u2i:         {_s_uid in _u2i}')
print(f'    Liked recipes:   {len(_s_rel)}')
print(f'    Unseen recipes:  {len(_s_unseen):,}')
_test_pred = _svd.predict(_u2i[_s_uid], _r2i[list(_s_unseen)[0]]).est
print(f'    Test prediction: {_test_pred:.3f}  ← should be a real number')


In [ ]:
# Step 4 — Method 1: Precision@k, Recall@k, F1@k
print('Step 4 — Evaluating Precision@k and Recall@k...')
K_VALUES  = [1, 3, 5, 10, 15, 20]
MAX_USERS = 300
eval_rows = []

for k in K_VALUES:
    p_l, r_l, f_l = [], [], []
    skipped = 0

    for uid in list(_relev.keys())[:MAX_USERS]:
        if uid not in _u2i:
            skipped += 1; continue

        ui     = _u2i[uid]
        seen   = set(df_inter[df_inter['user_id']==uid]['recipe_id'].values)

        # KEY FIX: use _all_recipes (original IDs), not unique_recipes (mapped indices)
        unseen = [r for r in _all_recipes if r not in seen and r in _r2i]
        if not unseen:
            skipped += 1; continue

        preds = sorted(
            [(r, _svd.predict(ui, _r2i[r]).est) for r in unseen],
            key=lambda x: x[1], reverse=True
        )
        rec_ids = [p[0] for p in preds[:max(K_VALUES)]]
        rel     = _relev[uid]

        p = prec_at_k(rec_ids, rel, k)
        r = rec_at_k(rec_ids, rel, k)
        p_l.append(p); r_l.append(r); f_l.append(f1(p, r))

    eval_rows.append({
        'k':         k,
        'method':    'CF (SVD)',
        'precision': np.mean(p_l) if p_l else 0,
        'recall':    np.mean(r_l) if r_l else 0,
        'f1':        np.mean(f_l) if f_l else 0,
    })
    print(f'  k={k:>2}  '
          f'P={np.mean(p_l):.4f}  '
          f'R={np.mean(r_l):.4f}  '
          f'F1={np.mean(f_l):.4f}  '
          f'(n={len(p_l)} users, skipped={skipped})')

eval_df = pd.DataFrame(eval_rows)


In [ ]:
# Step 5 — Method 2: Leave-one-out Hit@10
print('=== Method 2: Leave-One-Out Hit@10 ===')
hits = []

for uid in list(_relev.keys())[:200]:
    if uid not in _u2i: continue
    ui  = _u2i[uid]
    ur  = df_inter[df_inter['user_id']==uid].sort_values('rating', ascending=False)
    if len(ur) < 2: continue
    held = int(ur.iloc[0]['recipe_id'])
    if held not in _r2i: continue

    # KEY FIX: use _all_recipes (original IDs)
    unseen = [r for r in _all_recipes if r != held and r in _r2i]
    preds  = sorted(
        [(r, _svd.predict(ui, _r2i[r]).est) for r in unseen],
        key=lambda x: x[1], reverse=True
    )
    hits.append(int(held in {p[0] for p in preds[:10]}))

print(f'Hit@10: {np.mean(hits):.1%}  ({sum(hits)}/{len(hits)} users)')


In [ ]:
# Step 6 — Evaluation curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, metric in zip(axes, ['precision', 'recall', 'f1']):
    sub = eval_df[eval_df['method'] == 'CF (SVD)']
    ax.plot(sub['k'], sub[metric], marker='o', color=C_PURPLE,
             linewidth=2, markersize=6, label='CF (SVD)')
    ax.set_xlabel('k')
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f'{metric.capitalize()}@k', fontweight='bold')
    ax.set_ylim(0, max(sub[metric].max() * 1.3, 0.05))
    ax.legend(fontsize=8)

plt.suptitle('Recommender Evaluation — Food.com Dataset',
              fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/evaluation_curves.png', dpi=120, bbox_inches='tight')
plt.show()

print('\n=== Final Evaluation Summary ===')
k10 = eval_df[eval_df['k'] == 10].iloc[0]
print(f'Precision@10: {k10["precision"]:.4f}')
print(f'Recall@10:    {k10["recall"]:.4f}')
print(f'F1@10:        {k10["f1"]:.4f}')
print(f'Hit@10:       {np.mean(hits):.1%}')


---
## Section 13 — Hyperparameter Tuning (Optuna)
**Work Package: Hyperparameter Tuning**


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Use a small user sample for speed
_tune_users = list(_relev.keys())[:80]

def objective(trial):
    nf  = trial.suggest_int('n_factors',  10,  150)
    reg = trial.suggest_float('reg_all',   0.001, 1.0,  log=True)
    lr  = trial.suggest_float('lr_all',    0.001, 0.05, log=True)
    alp = trial.suggest_float('alpha',     0.0,   1.0)

    # Train SVD with trial params
    m = SVD(n_factors=nf, reg_all=reg, lr_all=lr, n_epochs=10, random_state=42)
    m.fit(_train)

    p_list = []
    for uid in _tune_users:
        if uid not in _u2i: continue
        ui  = _u2i[uid]
        rel = _relev.get(uid, set())
        if not rel: continue

        seen   = set(df_inter[df_inter['user_id']==uid]['recipe_id'].values)

        # KEY FIX: use _all_recipes (original IDs)
        unseen = [r for r in _all_recipes if r not in seen and r in _r2i]
        if not unseen: continue

        # CF scores
        cf_r = np.array([m.predict(ui, _r2i[r]).est for r in unseen])
        cf_n = (cf_r - cf_r.min()) / (cf_r.max() - cf_r.min() + 1e-9)

        # CB scores using user's top-rated recipes as proxy vector
        top_ids = df_inter[
            (df_inter['user_id']==uid) &
            (df_inter['rating']>=4)
        ]['recipe_id'].values[:5]
        vidx = [RID2IDX[r] for r in top_ids if r in RID2IDX]
        if not vidx: continue
        uv = R[vidx].mean(axis=0)

        unseen_ridx = [RID2IDX[r] for r in unseen if r in RID2IDX]
        if not unseen_ridx: continue
        cb_r = cosine_similarity(uv.reshape(1,-1), R[unseen_ridx]).flatten()
        cb_n = (cb_r - cb_r.min()) / (cb_r.max() - cb_r.min() + 1e-9)

        n     = min(len(cb_n), len(cf_n))
        final = alp * cb_n[:n] + (1 - alp) * cf_n[:n]
        rids  = [unseen[i] for i in np.argsort(final)[::-1][:10]]
        p_list.append(prec_at_k(rids, rel, 10))

    return np.mean(p_list) if p_list else 0.0


print('Running Optuna (50 trials)...')
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=50)

best = study.best_params
print(f'\nBest Precision@10: {study.best_value:.4f}')
for k, v in best.items():
    print(f'  {k:<15} = {v}')


In [ ]:
# Retrain with best params
print('Retraining with best params...')
best_svd = SVD(
    n_factors = best['n_factors'],
    reg_all   = best['reg_all'],
    lr_all    = best['lr_all'],
    n_epochs  = 30,
    random_state = 42
)
best_svd.fit(_train)
bp = best_svd.test(_test)
print(f'Tuned RMSE: {accuracy.rmse(bp, verbose=False):.4f}')
print(f'Tuned MAE:  {accuracy.mae(bp,  verbose=False):.4f}')

with open('models/svd_best.pkl', 'wb') as f:
    pickle.dump({'model': best_svd, 'params': best}, f)
print('Saved → models/svd_best.pkl')


In [ ]:
# Hyperparameter importance plots
tdf    = study.trials_dataframe()
params = ['params_n_factors','params_reg_all','params_lr_all','params_alpha']
labels = ['n_factors (k)', 'λ (reg)', 'lr', 'α (blend)']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, param, label in zip(axes, params, labels):
    if param not in tdf.columns: continue
    sc = ax.scatter(tdf[param], tdf['value'],
                     c=tdf['value'], cmap='RdYlGn', alpha=0.7, s=40)
    bv = best.get(param.replace('params_', ''), None)
    if bv is not None:
        ax.axvline(bv, color='black', linestyle='--',
                    linewidth=1.5, label=f'best={bv:.4f}')
        ax.legend(fontsize=7)
    ax.set_xlabel(label)
    ax.set_ylabel('Precision@10')
    ax.set_title(label, fontweight='bold')
    plt.colorbar(sc, ax=ax)

plt.suptitle('Optuna Hyperparameter Search', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/hyperparams.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 14 — Experiment Logging (W&B)
**Work Package: Experiments Logging**


In [ ]:
WANDB_ENABLED = False  # Set True after: pip install wandb && wandb login

if WANDB_ENABLED:
    import wandb
    for trial in study.trials:
        wandb.init(
            project = 'food-recommender-foodcom',
            name    = f'trial-{trial.number:03d}',
            config  = trial.params,
            reinit  = True
        )
        wandb.log({'precision_at_10': trial.value, **trial.params})
        wandb.finish()
    # Log best model metrics
    wandb.init(project='food-recommender-foodcom', name='best-model',
                config=best, tags=['final'])
    k10 = eval_df[eval_df['k']==10].iloc[0]
    wandb.log({
        'precision_at_10': k10['precision'],
        'recall_at_10':    k10['recall'],
        'f1_at_10':        k10['f1'],
        'hit_at_10':       np.mean(hits),
        'rmse':            accuracy.rmse(bp, verbose=False),
        **best
    })
    wandb.finish()
    print(f'{len(study.trials)} trials logged to W&B')
else:
    print('W&B disabled — set WANDB_ENABLED=True after: wandb login')
    print(f'Would log {len(study.trials)} trials')
    print(f'Best precision@10: {study.best_value:.4f}')
    print(f'Best params: {best}')


---
## Section 15 — Streamlit Frontend
**Work Package: Frontend Application**

Run with: `streamlit run app.py`


In [ ]:
APP = '''
import streamlit as st
import pandas as pd
import numpy as np
import pickle
from sklearn.metrics.pairwise import cosine_similarity
import plotly.express as px

st.set_page_config(
    page_title="Food Recommender",
    page_icon="\U0001f957",
    layout="wide"
)

FEATURE_MAX = dict(calories=2000, protein_g=150, carbs_g=300,
                    total_fat_g=100, sodium_mg=5000, sugar_g=200)
NUM   = list(FEATURE_MAX.keys())
LABEL = ["diabetic_ok","low_sodium","low_calorie","high_protein",
          "low_fat","high_fiber","heart_healthy","vegetarian",
          "vegan","gluten_free","dairy_free"]

@st.cache_data
def load_data():
    return pd.read_csv("data/recipes_clean.csv")

df = load_data()
nut = df[NUM].copy()
for c, mx in FEATURE_MAX.items():
    nut[c] = nut[c].fillna(0) / mx
R_app = pd.concat([nut, df[LABEL].fillna(0)], axis=1).values

st.title("\U0001f957 Health-Personalized Food Recommender")
st.caption(f"Food.com  |  {len(df):,} clean recipes")

c1, c2 = st.columns([1, 2])
with c1:
    st.subheader("Your health profile")
    cal  = st.slider("Target calories",   200, 800, 450, 50)
    prot = st.slider("Target protein (g)",  5,  80,  30,  5)
    carb = st.slider("Max carbs (g)",       10, 250, 120, 10)
    fat  = st.slider("Max fat (g)",          5,  80,  35,  5)
    sod  = st.slider("Max sodium (mg)",    100,2000, 600,100)
    sug  = st.slider("Max sugar (g)",        0,  50,  15,  5)
    st.divider()
    diab = st.checkbox("Type 2 Diabetes  (carbs<=45g, sugar<=10g)")
    hyp  = st.checkbox("Hypertension  (sodium<=600mg)")
    veg  = st.checkbox("Vegan")
    gf   = st.checkbox("Gluten-free")
    k    = st.slider("Number of recommendations", 3, 20, 8)

with c2:
    n  = np.array([cal/2000, prot/150, carb/300, fat/100, sod/5000, sug/200])
    l  = np.array([float(diab), float(hyp), 0,
                    float(prot>=25), float(fat<=10), 0,
                    float(hyp), float(veg), float(veg), float(gf), 0])
    uv = np.clip(np.concatenate([n, l]), 0, 1)
    sc = cosine_similarity(uv.reshape(1,-1), R_app).flatten()

    res = df.copy()
    res["score"] = sc
    res = res.sort_values("score", ascending=False)

    if diab: res = res[(res["carbs_g"] <= 45) & (res["sugar_g"] <= 10)]
    if hyp:  res = res[res["sodium_mg"] <= 600]
    if veg and "vegan" in res.columns:       res = res[res["vegan"] == 1]
    if gf  and "gluten_free" in res.columns: res = res[res["gluten_free"] == 1]

    recs = res.head(k).reset_index(drop=True)

    st.subheader(f"Top {k} recommendations")
    if len(recs) == 0:
        st.warning("No recipes match. Try relaxing some conditions.")
    else:
        fig = px.bar(
            recs, x="score", y="name", orientation="h",
            color="score", color_continuous_scale="Teal",
            hover_data=["calories","protein_g","carbs_g","sodium_mg"],
            labels={"score": "Match", "name": "Recipe"}
        )
        fig.update_layout(
            yaxis={"categoryorder": "total ascending"},
            height=400, showlegend=False, coloraxis_showscale=False
        )
        st.plotly_chart(fig, use_container_width=True)

        show_cols = [c for c in
            ["name","calories","protein_g","carbs_g",
             "total_fat_g","sodium_mg","sugar_g","minutes"]
            if c in recs.columns]
        st.dataframe(
            recs[show_cols].round(1),
            use_container_width=True, hide_index=True
        )

with st.sidebar:
    st.subheader("Dataset")
    st.metric("Clean recipes", f"{len(df):,}")
    if "diabetic_ok"  in df.columns: st.metric("Diabetic-ok",  f"{int(df.diabetic_ok.sum()):,}")
    if "vegan"        in df.columns: st.metric("Vegan",         f"{int(df.vegan.sum()):,}")
    if "gluten_free"  in df.columns: st.metric("Gluten-free",   f"{int(df.gluten_free.sum()):,}")
    if "heart_healthy"in df.columns: st.metric("Heart-healthy", f"{int(df.heart_healthy.sum()):,}")
'''

with open('app.py', 'w') as f:
    f.write(APP)
print('app.py written.  Run with:  streamlit run app.py')


---
## Section 16 — Full Pipeline Summary


In [ ]:
import glob

k10 = eval_df[eval_df['k'] == 10].iloc[0]

print('=' * 55)
print('FOOD RECOMMENDER — PIPELINE SUMMARY')
print('=' * 55)
print(f'Recipes raw:         {len(df_recipes_raw):>8,}')
print(f'Recipes clean:       {len(df):>8,}  ({len(df)/len(df_recipes_raw):.1%} kept)')
print(f'Interactions clean:  {len(df_inter):>8,}')
print(f'USDA enriched:       {df_usda["fiber_g"].notna().sum():>8,} recipes')
print(f'Recipe matrix R:     {str(R.shape):>8}')
print(f'Health labels:       {len(LABEL_COLS):>8}')
print()
print(f'CF RMSE:             {accuracy.rmse(_preds, verbose=False):>8.4f}')
print(f'CF MAE:              {accuracy.mae(_preds,  verbose=False):>8.4f}')
print(f'CF Precision@10:     {k10["precision"]:>8.4f}')
print(f'CF Recall@10:        {k10["recall"]:>8.4f}')
print(f'CF F1@10:            {k10["f1"]:>8.4f}')
print(f'LOO Hit@10:          {np.mean(hits):>8.1%}')
print(f'Best Optuna P@10:    {study.best_value:>8.4f}')
print(f'Best params:         {best}')
print()
plots = sorted(glob.glob('plots/*.png'))
print(f'Plots saved ({len(plots)}):')
for p in plots:
    print(f'  {p}')
print('=' * 55)
